# 数值稳定性和模型初始化

使用梯度下降进行神经网络的学习时，梯度的计算通过反向传播算法，从顶层（靠近输出层）梯度以**累乘**的方式计算到底层（靠近输入层）。

累乘会在较深的网络中带来两个严重问题：梯度爆炸和梯度消失，都会导致学习终止（无法获得有效梯度值）

- **梯度爆炸**：假设采用 ReLu 激活函数，底层的梯度约等于**所有更高层权值的乘积**（ReLu 导数只取 0 或 1）。如果更高层权值都比 1 大一点点，带来的结果（指数爆炸）会很容易超过浮点数上限（上溢）。

- **梯度消失**：假设采用 Sigmoid 激活函数，其**导数在绝对值较大的地方接近 0**。这样底层的梯度的累乘项中有很多更高层的 Sigmoid 导数。如果更高层权值都比较大，会导致累乘非常多趋近于 0 的数，带来的结果会很容易超过浮点数下限（下溢）。

通过动态调整学习率（学习率先大后小）也许能够控制深度学习的梯度爆炸和梯度消失问题，但给超参数的设计带来极大的麻烦和困难。

因此，需要有方法使梯度数值保持稳定：

1. 乘法变加法：如ResNet, LSTM
2. 归一化：梯度归一化、梯度裁剪
3. 合理的权重初始和激活函数

这里先讲权重初始，即模型初始化。我们使用称为 **Xavier初始化** 的方法。

此外，优化期间的注意和适当的正则化也可以进一步提高稳定性。

## Xavier初始化

需要先明确一个原理（概率论）：**使梯度数值保持稳定等价于使所有层梯度的方差尽量保持稳定值**。因为方差越大不确定性越大（分布越散），越容易出现不合理的梯度值。而根据反向传播算法计算的梯度的方差，与网络各层的**权重的方差**紧密相关（由反向传播计算过程决定）。

在前面的部分中，我们使用正态分布来初始化权重值。如果我们不指定初始化方法，
框架将使用默认的随机初始化方法，对于中等难度的问题，这种方法通常很有效。但对于较深的网络可能无法有效减小梯度失效的概率。

因此，Xavier初始化 的原则是：控制每层权重的期望和方差，**让每层的输出和权重保持期望为零、方差恒定**。

### 结论

不考虑激活函数推导。具体的结论为，对于任意一层网络权重，**该层的输入和输出维数**分别为 $ {n_{\text{in}}}, n_{\text{out}}$：

**如果想让输出方差等于输入方差（即 $\operatorname{Var}(y_j) = \sigma_x^2$），就必须有：**
$$
\sigma_w^2 = \frac{1}{n_{\text{in}}}
$$

**要让反向梯度方差也保持恒定（即前后层梯度方差相等），则需要：**
$$
\sigma_w^2 = \frac{1}{n_{\text{out}}}
$$

同时满足前向和反向的方差恒定条件几乎不可能（除非 $n_{\text{in}} = n_{\text{out}}$）。Xavier初始化（Glorot初始化）采用了一个折中方案，令：
$$
\sigma_w^2 = \frac{2}{n_{\text{in}} + n_{\text{out}}}
$$
这相当于在两种需求之间取得平衡，使得正向激活的方差和反向梯度的方差随层数变化都尽量平缓，不会出现指数级缩放。

对于均匀分布，Xavier初始化的区间为：
$$
\Big[-\frac{\sqrt{6}}{\sqrt{n_{\text{in}}+n_{\text{out}}}}, \frac{\sqrt{6}}{\sqrt{n_{\text{in}}+n_{\text{out}}}}\Big]
$$

对于正态分布，Xavier初始化的分布为：
$$
N\left(0, \frac{\sqrt{2}}{\sqrt{n_{\text{in}}+n_{\text{out}}}}\right)
$$

### 考虑激活函数

在 [让训练更加稳定.pdf](ppt/让训练更加稳定.pdf) 中有关于线性激活函数的推导，得到线性函数只能取 $f(x) = x$ 。

也就意味着如果激活函数在原点附近近似线性，就可以忽略其对 Xavier初始化 的影响，依然用之前的结论进行初始化。

可以通过泰勒展开进行分析：

$$
\operatorname{sigmoid}(x) = \frac{1}{2} + \frac{x}{4} - \frac{x^3}{48} + O(x^5)
$$

$$
\tanh(x) = 0 + x - \frac{x^3}{3} + O(x^5)
$$

$$
\operatorname{relu}(x) = 0 + x \quad \text{for } x \geq 0
$$

这告诉我们，在使用 sigmoid 作为激活函数时，需要调整 sigmoid，使其在原点附近线性：

$$
4 \times \operatorname{sigmoid}(x) - 2
$$

### 对称性

这里需要再提一个问题：能否将同一个隐藏层的所有参数初始化为$\mathbf{W}^{(1)} = c$，$c$为常量

在这种情况下，在前向传播期间，同一层隐藏单元将得到相同的输入和参数，
产生相同的激活，该激活被送到输出单元。
在反向传播期间，根据参数$\mathbf{W}^{(1)}$对输出单元进行微分，
得到一个梯度，其元素都取相同的值。
因此，在基于梯度的迭代（例如，小批量随机梯度下降）之后，
$\mathbf{W}^{(1)}$的所有元素仍然采用相同的值。
这样的迭代永远不会打破对称性，我们可能永远也无法实现网络的表达能力。
隐藏层的行为就好像只有一个单元。
请注意，虽然小批量随机梯度下降不会打破这种对称性，但暂退法正则化可以。

这种问题被称为神经网络设计中的**参数化所固有的对称性**。
假设我们有一个简单的多层感知机，它有一个隐藏层和两个隐藏单元。
在这种情况下，我们可以对第一层的权重$\mathbf{W}^{(1)}$进行重排列，
并且同样对输出层的权重进行重排列，可以获得相同的函数。
第一个隐藏单元与第二个隐藏单元没有什么特别的区别。
换句话说，**我们在每一层的隐藏单元之间具有排列对称性**。